In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertForTokenClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# FoodNER-based NER class
class FoodNER(nn.Module):
    def __init__(self, tokens_dim):
        super(FoodNER, self).__init__()

        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        # Use FoodNER base model for token classification
        self.pretrained = BertForTokenClassification.from_pretrained("Dizex/FoodBaseBERT-NER", num_labels=tokens_dim, ignore_mismatched_sizes=True)

    def forward(self, input_ids, attention_mask, labels=None):
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Dizex/FoodBaseBERT-NER")

In [2]:
import torch
from torch.utils.data import Dataset
import ast

class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Ensure 'Ingredient_Phrases' is a string
        text = str(row['Ingredient_Phrases'])

        # Convert 'Labels' from string representation to a list
        labels_list = ast.literal_eval(row['Labels'])  

        encoding = self.tokenizer(
            text.split(),  # Tokenize word by word
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            is_split_into_words=True  # Ensure token alignment with words
        )

        word_ids = encoding.word_ids(batch_index=0)  # Get token-word alignment
        label_ids = [-100] * self.max_len  # Default to ignored token

        # Assigning NER labels for each word token
        previous_word_id = None
        for i, word_id in enumerate(word_ids):
            if word_id is None or word_id >= len(labels_list):
                label_ids[i] = -100  # Ignore special tokens and out-of-range values
            elif word_id != previous_word_id:
                label_ids[i] = self.label_map.get(labels_list[word_id], -100)  # Assign correct label
            else:
                label_ids[i] = label_ids[i-1]  # Assign same label for subwords
            previous_word_id = word_id

        input_ids = encoding['input_ids'].squeeze(0)  # Remove batch dimension
        attention_mask = encoding['attention_mask'].squeeze(0)  # Remove batch dimension
        label_ids = torch.tensor(label_ids, dtype=torch.long)  # Convert to tensor

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_ids
        }

# Label mapping 
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DF': 5,
    'SIZE': 6,
    'O': -100  # Outside token or padding
}

# from sklearn.model_selection import train_test_split
# df = pd.read_csv('../../data/processed/10k_data_modified.csv')

- train + test split 

In [3]:
path = '../../data/augmented/'
train_df = pd.read_csv(path + 'aug_train_df.csv')
test_df = pd.read_csv(path + 'aug_test_df.csv')

In [4]:
train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


model = FoodNER(tokens_dim=len(label_map))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at Dizex/FoodBaseBERT-NER and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

/home/ritisha21089/miniconda3/lib/python3.9/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


FoodNER(
  (pretrained): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(28997, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, 

In [6]:
torch.cuda.set_device(1)  # Set GPU 1 as the active device
print("Now using GPU:", torch.cuda.current_device())

Now using GPU: 1


In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())
print("CUDA Device Name:", torch.cuda.get_device_name(0))
print("Current CUDA Device:", torch.cuda.current_device())

CUDA Available: True
CUDA Device Count: 2
CUDA Device Name: NVIDIA A100-PCIE-40GB
Current CUDA Device: 0


# MODEL TRAINING

- Training

In [6]:
from tqdm import tqdm

# Train
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm description with the latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)  # Store loss

    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/5, Average Loss: 0.1752


Epoch 2/5, Average Loss: 0.0626


Epoch 3/5, Average Loss: 0.0392


Epoch 4/5, Average Loss: 0.0294


Epoch 5/5, Average Loss: 0.0235


# EVALUATION

In [7]:
# Evaluation 
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)

In [8]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print("FoodBaseBERT-NER-augmentation results:")
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

FoodBaseBERT-NER-augmentation results:
Precision: 0.9631
Recall: 0.9727
F1-Score: 0.9677
              precision    recall  f1-score   support

           0     0.9946    0.9962    0.9954      5567
           1     0.9912    0.9880    0.9896      3985
           2     0.9884    0.9828    0.9856      9120
           3     0.8642    0.9389    0.9000      1457
           4     0.9851    0.9707    0.9778      4820
           5     0.9585    0.9818    0.9700       329
           6     0.9598    0.9508    0.9552       853

    accuracy                         0.9807     26131
   macro avg     0.9631    0.9727    0.9677     26131
weighted avg     0.9813    0.9807    0.9809     26131

